# Graph Intelligence: OpenAI Agents SDK + Neo4j

Three patterns for integrating Neo4j with the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/):
1. **MCP Agent** — AI agent that queries Neo4j using tools via the Model Context Protocol
2. **Custom Tools Agent** — combining MCP tools with hand-written Cypher-backed `@function_tool` definitions
3. **Memory Agent** — cross-session persistent memory via `neo4j-agent-memory`

The demo uses the publicly accessible **Neo4j companies knowledge graph** (`Organization` nodes linked to `Article` nodes via `[:MENTIONS]` relationships).

## 1. Setup

Install the required Python packages.

In [1]:
!pip install --quiet --upgrade openai-agents "neo4j-agent-memory[mcp]"
!pip install --ignore-requires-python --quiet neo4j-mcp-server
print("Packages installed ✓")

Packages installed ✓


## 2. Configuration

Set environment variables for Neo4j and OpenAI.

- The **companies demo database** is used for knowledge-graph queries (read-only, public).
- **Agent memory** requires a separate **writable** Neo4j instance. Set `MEMORY_NEO4J_*` env vars before running Section 6.

In [2]:
import os
from getpass import getpass

# OpenAI API key
os.environ.setdefault("OPENAI_API_KEY", getpass("OpenAI API key: "))

# Neo4j Knowledge Graph — read-only companies demo (public credentials)
os.environ["NEO4J_URI"]      = "neo4j+s://demo.neo4jlabs.com:7687"
os.environ["NEO4J_USERNAME"] = "companies"
os.environ["NEO4J_PASSWORD"] = "companies"
os.environ["NEO4J_DATABASE"] = "companies"

# Port for the neo4j-mcp-server HTTP listener
os.environ.setdefault("MCP_PORT", "8443")

# Neo4j Memory DB (requires WRITE access — replace with a real writable instance)
os.environ.setdefault("MEMORY_NEO4J_URI",      "neo4j+s://your-instance.databases.neo4j.io")
os.environ.setdefault("MEMORY_NEO4J_USERNAME",  "neo4j")
os.environ.setdefault("MEMORY_NEO4J_PASSWORD",  "your-password")
os.environ.setdefault("MEMORY_NEO4J_DATABASE",  "neo4j")

print("Configuration set.")
print("Neo4j URI: ", os.environ["NEO4J_URI"])
print("MCP Port:  ", os.environ["MCP_PORT"])
print("OpenAI key:", os.environ["OPENAI_API_KEY"][:8] + "...")

Configuration set.
Neo4j URI:  neo4j+s://demo.neo4jlabs.com:7687
MCP Port:   8443
OpenAI key: sk-svcac...


## 3. Neo4j MCP Server

Start `neo4j-mcp-server` in HTTP mode and verify the available tools.

> **Credentials note:** In HTTP mode the server extracts Neo4j credentials from the client's `Authorization: Basic ...` header — `NEO4J_USERNAME` / `NEO4J_PASSWORD` are **not** passed to the server process but are sent per-request by the MCP client.

In [3]:
import base64, copy, subprocess, time, asyncio
from agents import Agent, Runner
from agents.mcp import MCPServerStreamableHttp

# ── Patch: strip bogus required constraint from get-schema ─────────────────────
# neo4j-mcp-server v1.5.x advertises get-schema with required=["properties"] but
# the tool takes no inputs. The OpenAI Agents SDK validates inputs locally and
# rejects calls with empty args. This subclass removes the bogus constraint.
class PatchedMCPServerStreamableHttp(MCPServerStreamableHttp):
    """MCPServerStreamableHttp with get-schema required-field bug fixed."""
    async def list_tools(self, run_context=None, agent=None):
        tools = await super().list_tools(run_context, agent)
        patched = []
        for t in tools:
            if t.name == "get-schema":
                schema = getattr(t, "inputSchema", None) or {}
                if isinstance(schema, dict) and schema.get("required"):
                    t = copy.deepcopy(t)
                    t.inputSchema.pop("required", None)
            patched.append(t)
        return patched

# ── Start neo4j-mcp-server in HTTP mode ───────────────────────────────────────
# Credentials are NOT passed to the server — they arrive per-request via Basic Auth.
MCP_PORT = os.environ.get("MCP_PORT", "8443")
server_env = {k: v for k, v in os.environ.items()
              if k not in ("NEO4J_USERNAME", "NEO4J_PASSWORD")}

mcp_proc = subprocess.Popen(
    [
        "neo4j-mcp-server",
        "--neo4j-uri",            os.environ["NEO4J_URI"],
        "--neo4j-database",       os.environ["NEO4J_DATABASE"],
        "--neo4j-transport-mode", "http",
        "--neo4j-http-port",      MCP_PORT,
    ],
    env=server_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
print(f"neo4j-mcp-server started (PID {mcp_proc.pid}) on port {MCP_PORT}")
time.sleep(3)  # wait for server to bind
print("Server ready ✓")

neo4j-mcp-server started (PID 46826) on port 8443
Server ready ✓


In [4]:
# Build Basic Auth credentials for the MCP client
creds = base64.b64encode(
    f"{os.environ['NEO4J_USERNAME']}:{os.environ['NEO4J_PASSWORD']}".encode()
).decode()

mcp_server = PatchedMCPServerStreamableHttp(
    params={
        "url":     f"http://localhost:{MCP_PORT}/mcp",
        "headers": {"Authorization": f"Basic {creds}"},
    },
    name="neo4j",
)

# Connect once — the session stays alive for all subsequent cells
await mcp_server.connect()
tools = await mcp_server.list_tools()

print("Connected to Neo4j MCP ✓")
print("Available tools:")
for t in tools:
    print(f"  {t.name:30s} {t.description}")

Connected to Neo4j MCP ✓
Available tools:
  get-schema                     
		Retrieve the schema information from the Neo4j database, including node labels, relationship types, and property keys.
		If the database contains no data, no schema information is returned.
  list-gds-procedures            Use this tool to discover what graph science and analytics functions are available in the current Neo4j environment. It returns a structured list describing each function — what it does, how to use it, the inputs it needs, and what kind of results it produces. Do this before any reasoning, query generation, or analysis so you know what capabilities exist. Graph science and analytics functions help you with centrality, community detection, similarity, path finding, and identifying dependencies between nodes. The tool helps you understand the analytical capabilities of the system so that you can plan or compose the right graph science operations automatically. An empty response indicates that

## 4. MCP Agent

Connect to the Neo4j MCP server and run a multi-step agent query. The agent automatically selects which MCP tools to call (`get-schema`, `read-cypher`, etc.) to answer the user's question.

In [5]:
SYSTEM_PROMPT = """You are a graph database assistant. Your job is to answer user questions by querying Neo4j.
Always run 'get-schema' first if you are unfamiliar with the graph structure.
Use Cypher queries to retrieve data.
After running a query, provide a clear text summary of the results.
If the data is not found, state that clearly."""


async def run_mcp_agent(query: str):
    agent = Agent(
        name="neo4j_explorer",
        instructions=SYSTEM_PROMPT,
        mcp_servers=[mcp_server],
        model="gpt-4o",
    )
    print(f"Query: {query}\n")
    result = await Runner.run(agent, query)
    print(f"Result: {result.final_output}")
    return result


await run_mcp_agent("How many organizations are in the database?")

Query: How many organizations are in the database?

Result: There are 46,088 organizations in the database.


RunResult(input='How many organizations are in the database?', new_items=[ToolCallItem(agent=Agent(name='neo4j_explorer', handoff_description=None, tools=[], mcp_servers=[<__main__.PatchedMCPServerStreamableHttp object at 0x761d7e8e71d0>], mcp_config={}, instructions="You are a graph database assistant. Your job is to answer user questions by querying Neo4j.\nAlways run 'get-schema' first if you are unfamiliar with the graph structure.\nUse Cypher queries to retrieve data.\nAfter running a query, provide a clear text summary of the results.\nIf the data is not found, state that clearly.", prompt=None, handoffs=[], model='gpt-4o', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body

## 5. Custom Tools Agent

Combine the Neo4j MCP tools with hand-written Cypher-backed `@function_tool` definitions. Custom tools and MCP tools are passed to the same agent — the framework routes each call to the correct handler automatically.

In [6]:
from agents import function_tool
import neo4j as _neo4j

driver = _neo4j.AsyncGraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)

@function_tool
async def get_investments(company: str) -> list:
    """Returns the investments made by a company — a list of investment ids, names and types."""
    async with driver.session(database=os.environ["NEO4J_DATABASE"]) as session:
        result = await session.run(
            """MATCH (o:Organization)-[:HAS_INVESTOR]->(i)
               WHERE o.name = $company
               RETURN i.id AS id, i.name AS name, head(labels(i)) AS type""",
            company=company,
        )
        return [dict(r) async for r in result]

print("Custom tool defined: get_investments ✓")

Custom tool defined: get_investments ✓


In [7]:
async def run_custom_agent(query: str):
    agent = Agent(
        name="neo4j_custom",
        instructions="""You are a helpful assistant with access to a Neo4j graph database containing company data.
Use the available tools to answer questions about organizations, investments, and relationships.""",
        tools=[get_investments],
        mcp_servers=[mcp_server],
        model="gpt-4o",
    )
    print(f"Query: {query}\n")
    result = await Runner.run(agent, query)
    print(f"Result: {result.final_output}")
    return result


await run_custom_agent("Which companies did Google invest in?")

Query: Which companies did Google invest in?

Result: Google has invested in the following companies:

1. **Ionic Security**
2. **Avere Systems**
3. **FlexiDAO**
4. **Cloudflare**
5. **Trifacta**


RunResult(input='Which companies did Google invest in?', new_items=[ToolCallItem(agent=Agent(name='neo4j_custom', handoff_description=None, tools=[FunctionTool(name='get_investments', description='Returns the investments made by a company — a list of investment ids, names and types.', params_json_schema={'properties': {'company': {'title': 'Company', 'type': 'string'}}, 'required': ['company'], 'title': 'get_investments_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x761d7c618950>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)], mcp_servers=[<__main__.PatchedMCPServerStreamableHttp object at 0x761d7e8e71d0>], mcp_config={}, instructions='You are a helpful assistant with access to a Neo4j graph database containing company d

## 6. Memory Agent

[`neo4j-agent-memory`](https://pypi.org/project/neo4j-agent-memory/) stores agent knowledge as a graph in Neo4j, enabling cross-session persistence via hybrid retrieval.

The pattern implements two hooks around each agent call:
- **Before hook (`inject_memory_context`):** retrieves relevant memories from Neo4j and injects them into the system prompt so the model has prior context.
- **After hook (`save_interaction`):** saves the interaction to the memory graph so future sessions can recall it.

The two-turn demo shows memory in action: Turn 2 automatically recalls context set in Turn 1.

> **Prerequisite:** Memory requires a **writable** Neo4j instance. Set `MEMORY_NEO4J_*` env vars in Section 2.

In [8]:
from neo4j_agent_memory import MemoryIntegration

MEMORY_SYSTEM_PROMPT = """You are a helpful assistant with access to a Neo4j graph database AND
long-term memory of past interactions.

YOUR PRIORITY:
1. Check the MEMORY CONTEXT section below first. Use it directly if relevant.
2. If the answer is not in memory, query Neo4j using the available tools.
3. State clearly if the answer cannot be found."""


async def setup_memory():
    """Initialise and return a MemoryIntegration, or None if unavailable."""
    mem_uri = os.environ.get("MEMORY_NEO4J_URI", "")
    if "your-instance" in mem_uri:
        print("Memory DB not configured — set MEMORY_NEO4J_URI in Section 2.")
        print("Continuing without memory persistence.\n")
        return None
    # neo4j-agent-memory bundles driver v5 which requires bolt+ssc://
    mem_uri = mem_uri.replace("neo4j+s://", "bolt+ssc://")
    try:
        m = MemoryIntegration(
            neo4j_uri=mem_uri,
            neo4j_user=os.environ.get("MEMORY_NEO4J_USERNAME", "neo4j"),
            neo4j_password=os.environ.get("MEMORY_NEO4J_PASSWORD", ""),
            neo4j_database=os.environ.get("MEMORY_NEO4J_DATABASE", "neo4j"),
        )
        await m.connect()
        print("Memory service initialised ✓")
        return m
    except Exception as e:
        print(f"Memory unavailable ({e}) — continuing without persistence.")
        return None


# ── BEFORE hook: retrieve relevant memories, inject into system prompt ─────────
async def inject_memory_context(mem, user_query: str) -> str:
    if mem is None:
        return MEMORY_SYSTEM_PROMPT
    try:
        ctx = await mem.get_context(query=user_query, max_items=3)
        memories = ctx.get("short_term", []) + ctx.get("long_term", [])
        if memories:
            print(f"  ↳ Injecting {len(memories)} memor{'y' if len(memories) == 1 else 'ies'} into context.")
            for i, m in enumerate(memories[:3]):
                print(f"    Memory {i + 1}: {str(m.get('content', m))[:100]}...")
            ctx_text = "\n".join(f"- {m.get('content', str(m))}" for m in memories)
            return f"{MEMORY_SYSTEM_PROMPT}\n\n--- MEMORY CONTEXT ---\n{ctx_text}\n----------------------"
    except Exception:
        pass
    return MEMORY_SYSTEM_PROMPT


# ── AFTER hook: save interaction to memory graph ───────────────────────────────
async def save_interaction(mem, user_query: str, response: str):
    if mem is None:
        return
    try:
        await mem.store_message("user",      user_query)
        await mem.store_message("assistant", response)
        print("  [Hook] Interaction saved to Neo4j memory graph.")
    except Exception as e:
        print(f"  [Hook] Save failed: {e}")


# ── Run agent with memory hooks ────────────────────────────────────────────────
async def run_memory_demo():
    mem = await setup_memory()

    async def run_turn(query: str):
        print(f"\n[USER]: {query}")
        system_with_ctx = await inject_memory_context(mem, query)
        agent = Agent(
            name="neo4j_analyst",
            instructions=system_with_ctx,
            mcp_servers=[mcp_server],
            model="gpt-4o",
        )
        result = await Runner.run(agent, query)
        print(f"[AGENT]: {result.final_output}")
        await save_interaction(mem, query, result.final_output)
        return result

    # Turn 1: establish research context
    await run_turn(
        "I am conducting a competitive analysis of 'Google'. I am specifically "
        "worried about their subsidiaries and who their top-tier competitors are in the AI space."
    )

    print("\n--- Indexing memory (5s)... ---\n")
    await asyncio.sleep(5)

    # Turn 2: follow-up — Turn 1 context is injected automatically
    await run_turn(
        "What are the main risks in the supply chain for the company I am currently tracking?"
    )

    if mem:
        await mem.close()


await run_memory_demo()

/home/codespace/.python/current/lib/python3.12/site-packages/fastmcp/server/auth/providers/jwt.py:10: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import JsonWebKey, JsonWebToken


Memory DB not configured — set MEMORY_NEO4J_URI in Section 2.
Continuing without memory persistence.


[USER]: I am conducting a competitive analysis of 'Google'. I am specifically worried about their subsidiaries and who their top-tier competitors are in the AI space.
[AGENT]: It seems there was an issue accessing the database directly. Let me try gathering some information for you another way.

For a competitive analysis of Google in the AI space, you might look into:

1. **Google's Subsidiaries**: Some major subsidiaries involved in AI include DeepMind, Google Brain, and Waymo.

2. **Top-tier Competitors**: In AI, some of Google's major competitors include:
   - **Microsoft**: With Azure AI and partnership with OpenAI.
   - **Amazon**: Through AWS AI services.
   - **IBM**: Known for Watson.
   - **Meta (Facebook)**: Investing heavily in AI research.
   - **Apple**: Developing AI for consumer products.

If you need detailed data or want to explore more through the database, we might

## 7. Summary

| Pattern | Key APIs | Use Case |
|---------|----------|----------|
| MCP Agent | `MCPServerStreamableHttp`, `Agent`, `Runner` | Natural-language Neo4j queries via MCP |
| Custom Tools Agent | `@function_tool`, `Agent(tools=[...], mcp_servers=[...])` | Combine MCP tools with custom Cypher logic |
| Memory Agent | `MemoryIntegration`, before/after hooks | Cross-session persistent knowledge |

### Key Implementation Notes

- **`neo4j-mcp-server` HTTP mode** — credentials are passed per-request via `Authorization: Basic ...` header, not as server env vars.
- **`PatchedMCPServerStreamableHttp`** — works around a v1.5.x bug where `get-schema` incorrectly declares `required: ["properties"]`. Can be removed once the bug is fixed upstream.
- **`bolt+ssc://`** — required for `neo4j-agent-memory` (bundles driver v5, no `neo4j+s://` routing).
- **`asyncio.run()`** — wraps async agent code for clean notebook execution.

### Resources

- [OpenAI Agents SDK Documentation](https://openai.github.io/openai-agents-python/)
- [OpenAI Agents SDK — MCP Support](https://openai.github.io/openai-agents-python/mcp/)
- [neo4j-mcp-server on PyPI](https://pypi.org/project/neo4j-mcp-server/)
- [neo4j-agent-memory on PyPI](https://pypi.org/project/neo4j-agent-memory/)
- [Neo4j Python Driver Documentation](https://neo4j.com/docs/python-manual/current/)